# 2D cylinder flow (PhiFlow grid) + closed-loop swarm control

In [1]:
# Environment (no-op when already satisfied; restart the kernel if pip changes anything).
%pip install -q "phiflow==3.4.0" "phiml==1.14.1" "jax[cpu]" numpyro equinox meshio matplotlib scipy tqdm
# dynestyx is used from a local checkout:
# %pip install -q -e ~/Documents/GitHub/dynestyx


Note: you may need to restart the kernel to use updated packages.


In [2]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
from jax import random as jr

In [3]:
from pathlib import Path
import subprocess
import sys

import phi
from phi.flow import *
from tqdm.notebook import trange

assert phi.__version__.startswith("3."), (
    f"PhiFlow 3.x required (verified on 3.4.0); found {phi.__version__}"
)

# The finite-volume pressure solve needs double precision on this mesh.
math.set_global_precision(64)


In [7]:
@jit_compile_linear
def momentum_eq(u, u_prev, dt, diffusivity=0.01):
    diffusion_term = dt * diffuse.differential(u, diffusivity, correct_skew=False)
    advection_term = dt * advect.differential(u, u_prev, order=1)
    return u + advection_term + diffusion_term

@jit_compile
def implicit_time_step(v, dt):
    v = math.solve_linear(momentum_eq, v, Solve(x0=v), u_prev=v, dt=-dt)
    # The right-hand side can be nearly zero; a small absolute tolerance
    # avoids rejecting a valid direct solve because of roundoff.
    v, p = fluid.make_incompressible(v, (), Solve('scipy-direct', abs_tol=1e-8))
    return v


# Part I — Flow and particle advection

Incompressible Navier–Stokes on a $128 \times 64$ staggered grid over $[0,16] \times [0,8]$, with a cylinder of radius $R = 1$ at $(4, 4.05)$ as an obstacle and two particles advected by the flow:

$$
\begin{aligned}
&\partial_t \mathbf v + (\mathbf v \cdot \nabla)\mathbf v = -\nabla p + \nu \nabla^2 \mathbf v, \qquad \nabla \cdot \mathbf v = 0\\
&\frac{d \mathbf{x}_i}{dt} = \mathbf v(\mathbf x_i, t) + \mathbf u_i
\end{aligned}
$$


The state  $\{\mathbf v, \mathbf x_1, \mathbf x_2\}$, declared through a `Layout`; the PhiFlow step runs inside `jax.pure_callback` (as in the PhiFlow Wake_Flow example).

In [10]:
import dynestyx as dsx
from dynestyx import (
    DiracInitialCondition,
    DiracObservation,
    GaussianObservation,
    DiracStateEvolution,
    DynamicalModel,
    Layout,
)

In [11]:
# Grid version: staggered grid + semi-Lagrangian advection, cylinder as obstacle.
from jax.scipy.ndimage import map_coordinates

NX, NY = 128, 64
LX, LY = 16., 8.
dx, dy = LX / NX, LY / NY
bounds = Box(x=LX, y=LY)

U = 5.0                                        # inflow speed
R = 1.0                                        # cylinder radius (D = 2R); keep D / LY <~ 0.3 to limit wall blockage
Re = 1000.0                                       # Re = U D / nu; above ~1000 numerical diffusion dominates on this grid
cyl_center = jnp.array([4.0, 4.05])            # slightly off-centre to trigger shedding
cylinder = Sphere(x=float(cyl_center[0]), y=float(cyl_center[1]), radius=R)
boundary = {'x-': vec(x=U, y=0), 'x+': ZERO_GRADIENT, 'y': 0}
viscosity = U * 2 * R / Re

def to_field(V):
    """(NX+1, NY+1, 2) padded staggered array -> StaggeredGrid"""
    return StaggeredGrid(tensor(V, spatial('x,y'), channel(vector='x,y')), boundary, x=NX, y=NY, bounds=bounds)

def to_array(v):
    return jnp.asarray(v.staggered_tensor().numpy('x,y,vector'))

@jit_compile
def fluid_step(v, dt):
    v = advect.semi_lagrangian(v, v, dt)
    v = diffuse.explicit(v, viscosity, dt)
    v, p = fluid.make_incompressible(v, cylinder, Solve('scipy-direct'))
    return v

def interpolate(V, x):
    """Bilinear interpolation of the staggered velocity at x: (2,), pure JAX.
    V[i, j, 0] = v_x at (i dx, (j+1/2) dy),  V[i, j, 1] = v_y at ((i+1/2) dx, j dy)."""
    vx = map_coordinates(V[:, :NY, 0], [x[0] / dx, x[1] / dy - 0.5], order=1, mode='nearest')
    vy = map_coordinates(V[:NX, :, 1], [x[0] / dx - 0.5, x[1] / dy], order=1, mode='nearest')
    return jnp.stack([vx, vy])

PARTICLES = ('particle_1', 'particle_2')   # state keys
CONTROLS = ('u_1', 'u_2')                         # matching control keys


def advect_particle(V, x, u, dt):
    """One Euler step for one particle; a particle inside the cylinder is stuck."""
    inside = jnp.linalg.norm(x - cyl_center) < R
    return x + dt * jnp.where(inside, 0.0, interpolate(V, x) + u)


def make_stepper(dt):

    def step_phi(V):
        return to_array(fluid_step(to_field(V), dt))

    def stepper(state, u, t_now, t_next):
        V = state['velocity']
        V = jax.pure_callback(step_phi, jax.ShapeDtypeStruct(V.shape, V.dtype), V, vmap_method="sequential")
        new = {'velocity': V}
        for name in PARTICLES:
            new[name] = advect_particle(V, state[name], 0.0, dt)
        return new

    return stepper

def observation_function(state, u, t):
    return state

dt = 0.05

In [12]:
v0, _ = fluid.make_incompressible(StaggeredGrid(vec(x=U, y=0), boundary, x=NX, y=NY, bounds=bounds), cylinder, Solve('scipy-direct'))
starts = ([1.0, 3.0], [1.0, 4.3], [1.0, 5.5])
state_0 = {'velocity': to_array(v0), **{name: jnp.array(x) for name, x in zip(PARTICLES, starts)}}
step = make_stepper(dt)

out = step(state_0, None, 0.0, dt)

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3314592193.py:51: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [13]:
state_layout = Layout.from_example(state_0)
observation_layout = Layout.from_example(observation_function(state_0, None, 0.0))

In [14]:
initial_condition = DiracInitialCondition(state_0, state_layout=state_layout)
evolution = DiracStateEvolution(F=step, state_layout=state_layout)
observation_model = DiracObservation( observation_function, state_layout=state_layout, 
                                        observation_layout=observation_layout)
dynamics = DynamicalModel(
        initial_condition=initial_condition,
        state_evolution=evolution,
        observation_model=observation_model,
        state_layout=state_layout,
        observation_layout=observation_layout,
    )

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3314592193.py:51: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [15]:
T = 25.0  # shedding develops around t ~ 25
times = jnp.arange(0.0, T, step=dt)

result = dsx.simulate(
    dynamics,
    rng_key=jr.key(13),
    predict_times=times,
)
states = result.states
assert jnp.isfinite(states["velocity"]).all()

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3314592193.py:51: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [16]:
v_warm = states["velocity"][0, -1]   # used as warm start for the controlled simulation
v_warm.shape

(129, 65, 2)

In [17]:
# # Video: vorticity + particle trajectory
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
# from IPython.display import HTML
# plt.rcParams["animation.embed_limit"] = 100  # MB
# # LaTeX look without a LaTeX install: matplotlib's Computer Modern fonts.
# # (text.usetex=True needs pdflatex + dvipng, and re-runs LaTeX for every frame.)
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.serif": ["cmr10", "DejaVu Serif"],
#     "mathtext.fontset": "cm",
#     "axes.formatter.use_mathtext": True,
#     "axes.unicode_minus": False,
# })

# V = np.asarray(states["velocity"]).reshape(-1, *state_0["velocity"].shape)   # (T, NX+1, NY+1, 2)
# X = np.stack([np.asarray(states[name]).reshape(-1, 2) for name in PARTICLES], axis=1)  # (T, 3, 2)
# labels = [rf"particle {name[-1]} ($y_0 = {y}$)" for name, (_, y) in zip(PARTICLES, starts)]
# colors = ["gold", "limegreen", "magenta"]
# u_c = 0.5 * (V[:, 1:, :NY, 0] + V[:, :-1, :NY, 0])                           # cell-centred v_x
# v_c = 0.5 * (V[:, :NX, 1:, 1] + V[:, :NX, :-1, 1])                           # cell-centred v_y
# vort = np.gradient(v_c, dx, axis=1) - np.gradient(u_c, dy, axis=2)          # (T, NX, NY)
# vmax = np.percentile(np.abs(vort), 99)

# # Rendering only: upsample the field in space, and optionally in time.
# SMOOTH = 3        # spatial upsampling factor (cubic spline), rendering only
# STEP = 1.0        # stored steps per rendered frame; < 1 interpolates extra frames
# FRAME_DT = float(np.asarray(times)[1] - np.asarray(times)[0]) * STEP   # simulated time per frame
# FPS = 1.0 / FRAME_DT                                                   # real time: 1 s of video per time unit
# from scipy.ndimage import zoom

# def at(i_f, arr):   # linear interpolation in time at a fractional frame index
#     i0 = int(np.floor(i_f)); i1 = min(i0 + 1, len(arr) - 1); a = i_f - i0
#     return (1 - a) * arr[i0] + a * arr[i1]

# fig, ax = plt.subplots(figsize=(10, 5))
# im = ax.imshow(zoom(vort[0], SMOOTH, order=3).T, origin="lower", extent=(0, LX, 0, LY),
#                cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="bilinear")
# ax.add_patch(plt.Circle(cylinder.center.numpy("vector"), float(cylinder.radius), color="k"))
# trails = [ax.plot([], [], "-", lw=1, color=c)[0] for c in colors]
# dots = [ax.plot([], [], "o", color=c, mec="k", ms=7, label=l)[0] for c, l in zip(colors, labels)]
# ax.legend(loc="upper right", fontsize=8)
# fig.colorbar(im, ax=ax, label=r"vorticity $\omega$", shrink=0.8)
# ax.set_xlim(0, LX); ax.set_ylim(0, LY)
# ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$y$")

# def update(i_f):
#     im.set_data(zoom(at(i_f, vort), SMOOTH, order=3).T)
#     x_now, k = at(i_f, X), int(np.ceil(i_f)) + 1
#     for p, (trail, dot) in enumerate(zip(trails, dots)):
#         trail.set_data(X[:k, p, 0], X[:k, p, 1])
#         dot.set_data([x_now[p, 0]], [x_now[p, 1]])
#     ax.set_title(rf"$t = {at(i_f, np.asarray(times)):.1f}$")
#     return (im, *trails, *dots)

# anim = FuncAnimation(fig, update, frames=np.arange(0, len(X) - 1, STEP),
#                      interval=1000 * FRAME_DT, blit=True)
# plt.close(fig)
# anim.save("wake_uncontrolled.mp4", fps=FPS, dpi=120)   # needs ffmpeg; use a .gif filename for the pillow writer
# HTML(anim.to_jshtml())

# Part II: Linear feedback control

Each particle now carries a control $\mathbf u_i$ that adds to the flow velocity:

$$\frac{d \mathbf{x}_i}{dt} = \mathbf v(\mathbf x_i, t) + \mathbf u_i$$

With target $\mathbf x^\star = (4,4)$ and error $\mathbf e_i = \mathbf x^\star - \mathbf x_i$, the policy is a PID law

$$\mathbf u_i = K_p \mathbf e_i + K_i \int_0^t \mathbf e_i \, ds$$

In [18]:
from dynestyx.inference.configs.filter import EnKFConfig, PFConfig

In [19]:
import equinox as eqx

target = jnp.array([4.0, 4.0])  # target position for the particles: inside the cylinder, so unreachable

# Which PID terms each particle keeps: (P, I, D).
TERMS = {'particle_1': (0.5, 0., 0.),    # linear feedback only
         'particle_2': (0.4, 0.1, 0.),    # linear feedback + history
        } 


class PIDController(eqx.Module):
    Kp: float
    Ki: float = 0.0
    Kd: float = 0.0

    def __call__(self, x_hat, t_now, t_next, s):
        cum_error, prev_error = s
        x = state_layout.unflatten(x_hat.mean)   # here we only use the mean of the distribution
        h = t_next - t_now
        u, cum_next, prev_next = {}, {}, {}
        for name, u_name in zip(PARTICLES, CONTROLS):
            p, i, d = TERMS[name]
            e = target - x[name]
            cum = cum_error[name] + e * h
            de = (e - prev_error[name]) / h
            u[u_name] = p * self.Kp * e + i * self.Ki * cum + d * self.Kd * de
            cum_next[name], prev_next[name] = cum, e
        return u, (cum_next, prev_next)


In [20]:
policy = PIDController(Kp=1.0, Ki=1.0, Kd=0.0)

# The policy returns its control in this structure; the stepper receives the same pytree.
control_layout = Layout.from_example({u_name: jnp.zeros(2) for u_name in CONTROLS})


In [21]:
def make_stepper(dt):

    def step_phi(V):
        return to_array(fluid_step(to_field(V), dt))

    def stepper(state, u, t_now, t_next):
        V = state['velocity']
        V = jax.pure_callback(step_phi, jax.ShapeDtypeStruct(V.shape, V.dtype), V, vmap_method="sequential")
        new = {'velocity': V}
        for name, u_name in zip(PARTICLES, CONTROLS):
            new[name] = advect_particle(V, state[name], u[u_name], dt)
        return new

    return stepper


v0, _ = fluid.make_incompressible(StaggeredGrid(vec(x=U, y=0), boundary, x=NX, y=NY, bounds=bounds), cylinder, Solve('scipy-direct'))

state_0 = {'velocity': v_warm, **{name: jnp.array([12.0, 6.0]) for name in PARTICLES}}
step = make_stepper(dt)

# (cumulative error, previous error) per particle; no derivative kick on the first step
s_0 = ({name: jnp.zeros(2) for name in PARTICLES},
       {name: target - state_0[name] for name in PARTICLES})

initial_condition = DiracInitialCondition(state_0, state_layout=state_layout)
evolution = DiracStateEvolution(F=step, state_layout=state_layout)


def observation_function(state, u, t):   # only the particle positions, not the velocity field
    return {name: state[name] for name in PARTICLES}

observation_layout = Layout.from_example(observation_function(state_0, None, 0.0))

# observation_model = DiracObservation( observation_function, state_layout=state_layout, 
#                                         observation_layout=observation_layout)
observation_model = GaussianObservation( observation_function, state_layout=state_layout,
                                        observation_layout=observation_layout, cov = 1e-8)


In [22]:
dynamics = DynamicalModel(
        initial_condition=initial_condition,
        state_evolution=evolution,
        observation_model=observation_model,
        state_layout=state_layout,
        observation_layout=observation_layout,
        control_layout=control_layout,
    )

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3938603066.py:4: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [23]:
T = 10.0 
times = jnp.arange(0.0, T, step=dt)

results = dsx.simulate(
    dynamics,
    rng_key=jr.key(13),
    predict_times=times,
    control_policy=policy,
    filter_config=PFConfig(filter_source="cuthbert", record_filtered_states_mean=True, n_particles=1),
    initial_policy_state =s_0
)

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3938603066.py:4: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [24]:
states = results.states

In [25]:
assert all(jnp.isfinite(states[name]).all() for name in states.keys())

In [26]:
# # Video: vorticity + particle trajectory
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
# from IPython.display import HTML
# plt.rcParams["animation.embed_limit"] = 100  # MB
# # LaTeX look without a LaTeX install: matplotlib's Computer Modern fonts.
# # (text.usetex=True needs pdflatex + dvipng, and re-runs LaTeX for every frame.)
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.serif": ["cmr10", "DejaVu Serif"],
#     "mathtext.fontset": "cm",
#     "axes.formatter.use_mathtext": True,
#     "axes.unicode_minus": False,
# })

# V = np.asarray(states["velocity"]).reshape(-1, *state_0["velocity"].shape)   # (T, NX+1, NY+1, 2)
# X = np.stack([np.asarray(states[name]).reshape(-1, 2) for name in PARTICLES], axis=1)  # (T, 3, 2)
# terms = {'particle_1': "P only", 'particle_2': "P + I", 'particle_3': "P + I + D"}
# labels = [f"particle {name[-1]}: {terms[name]}" for name in PARTICLES]
# colors = ["gold", "limegreen", "magenta"]
# u_c = 0.5 * (V[:, 1:, :NY, 0] + V[:, :-1, :NY, 0])                           # cell-centred v_x
# v_c = 0.5 * (V[:, :NX, 1:, 1] + V[:, :NX, :-1, 1])                           # cell-centred v_y
# vort = np.gradient(v_c, dx, axis=1) - np.gradient(u_c, dy, axis=2)          # (T, NX, NY)
# vmax = np.percentile(np.abs(vort), 99)

# # Rendering only: upsample the field in space, and optionally in time.
# SMOOTH = 3        # spatial upsampling factor (cubic spline), rendering only
# STEP = 1.0        # stored steps per rendered frame; < 1 interpolates extra frames
# FRAME_DT = float(np.asarray(times)[1] - np.asarray(times)[0]) * STEP   # simulated time per frame
# FPS = 1.0 / FRAME_DT                                                   # real time: 1 s of video per time unit
# from scipy.ndimage import zoom

# def at(i_f, arr):   # linear interpolation in time at a fractional frame index
#     i0 = int(np.floor(i_f)); i1 = min(i0 + 1, len(arr) - 1); a = i_f - i0
#     return (1 - a) * arr[i0] + a * arr[i1]

# fig, ax = plt.subplots(figsize=(10, 5))
# im = ax.imshow(zoom(vort[0], SMOOTH, order=3).T, origin="lower", extent=(0, LX, 0, LY),
#                cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="bilinear")
# ax.add_patch(plt.Circle(cylinder.center.numpy("vector"), float(cylinder.radius), color="k"))
# trails = [ax.plot([], [], "-", lw=1, color=c)[0] for c in colors]
# dots = [ax.plot([], [], "o", color=c, mec="k", ms=7, label=l)[0] for c, l in zip(colors, labels)]
# ax.legend(loc="upper right", fontsize=8)
# fig.colorbar(im, ax=ax, label=r"vorticity $\omega$", shrink=0.8)
# ax.set_xlim(0, LX); ax.set_ylim(0, LY)
# ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$y$")

# def update(i_f):
#     im.set_data(zoom(at(i_f, vort), SMOOTH, order=3).T)
#     x_now, k = at(i_f, X), int(np.ceil(i_f)) + 1
#     for p, (trail, dot) in enumerate(zip(trails, dots)):
#         trail.set_data(X[:k, p, 0], X[:k, p, 1])
#         dot.set_data([x_now[p, 0]], [x_now[p, 1]])
#     ax.set_title(rf"$t = {at(i_f, np.asarray(times)):.1f}$")
#     return (im, *trails, *dots)

# anim = FuncAnimation(fig, update, frames=np.arange(0, len(X) - 1, STEP),
#                      interval=1000 * FRAME_DT, blit=True)
# plt.close(fig)
# anim.save("wake_controlled.mp4", fps=FPS, dpi=120)   # needs ffmpeg; use a .gif filename for the pillow writer
# HTML(anim.to_jshtml())

# Part III Interacting particles 

$N = 50$ particles, each with its own control, started as three Gaussian blobs. The control adds a pairwise interaction and a noise term to the same feedback law:

$$\mathbf u_i = K_p \mathbf e_i + K_i \int_0^t \mathbf e_i \, ds + K_s \sum_{j \neq i} \mathbf F(\mathbf x_i - \mathbf x_j) + \sigma\, \boldsymbol\xi_i, \qquad \boldsymbol\xi_i \sim \mathcal N(0, I)$$

The interaction is the gradient of a Morse potential, $U(r) = C_r e^{-r/\ell_r} - C_a e^{-r/\ell_a}$:

$$\mathbf F(\mathbf d) = \left[ \frac{C_r}{\ell_r} e^{-r/\ell_r} - \frac{C_a}{\ell_a} e^{-r/\ell_a} \right] \frac{\mathbf d}{r}, \qquad r = \sqrt{\|\mathbf d\|^2 + \varepsilon^2}$$

In [27]:
from dynestyx.inference.configs.filter import EnKFConfig, PFConfig

In [28]:
n_particles = 50
key = jr.PRNGKey(0)
# Three Gaussian blobs: particles are split as evenly as possible between the centres.
locs = jnp.array([[7.0, 2.5], [12.0, 4.0], [14.0, 6.0]])
blob_scale = 1.0
blob_of = jnp.arange(n_particles) % len(locs)
particles_0 = jr.normal(key, shape=(n_particles, 2)) * blob_scale + locs[blob_of]

v0, _ = fluid.make_incompressible(StaggeredGrid(vec(x=U, y=0), boundary, x=NX, y=NY, bounds=bounds), cylinder, Solve('scipy-direct'))
state_0 = {'velocity': v_warm, 'particles' : particles_0}

state_layout = Layout.from_example(state_0)

In [29]:
particles_0.shape

(50, 2)

In [30]:
from os import name
import equinox as eqx

target = jnp.array([4.0, 4.0])  # target position for the particles: inside the cylinder, so unreachable

# Attraction-repulsion between boats: the gradient of a Morse potential
#   U(r) = C_r exp(-r/l_r) - C_a exp(-r/l_a),
# repulsive at short range (C_r > C_a) and attractive at long range (l_r < l_a).
# The pair force vanishes at the equilibrium spacing d, where
#   (C_r/l_r) exp(-d/l_r) = (C_a/l_a) exp(-d/l_a)   ->   d ~ 0.81 for the values below.
C_r, C_a = 1.5, 0.5     # strengths
l_r, l_a = 0.3, 1.0     # ranges
EPS = 1e-3              # softening: keeps r > 0 for coincident boats


def swarm_force(x):
    """u_swarm,i = -grad_i sum_j U(|x_i - x_j|); x: (N, 2) -> (N, 2)."""
    d = x[:, None, :] - x[None, :, :]                        # (N, N, 2), points i away from j
    r = jnp.sqrt(jnp.sum(d ** 2, axis=-1) + EPS ** 2)        # (N, N), softened
    mag = C_r / l_r * jnp.exp(-r / l_r) - C_a / l_a * jnp.exp(-r / l_a)
    f = (mag / r)[..., None] * d                             # > 0 pushes i away from j
    f = f * (1.0 - jnp.eye(x.shape[0]))[..., None]           # no self-interaction
    return jnp.sum(f, axis=1)


class Controller(eqx.Module):
    Kp: float
    Ki: float = 0.0
    Kd: float = 0.0
    Ks: float = 1.0      # weight on the attraction-repulsion term

    def __call__(self, x_hat, t_now, t_next, s):
        cum_error , key = s
        x = state_layout.unflatten(x_hat.mean)   # here we only use the mean of the distribution
        h = t_next - t_now

        particles = x["particles"]

        e = target[None] - particles
        cum = cum_error + e * h

        key, subkey = jr.split(key)
        random_force = jr.normal(subkey, shape=particles.shape) * 0.2

        u = self.Kp * e + self.Ki * cum + self.Ks * swarm_force(particles) + random_force

        return u, (cum, key)


In [31]:
# (cumulative error, previous error) per particle; no derivative kick on the first step
s_0 = (jnp.zeros_like(particles_0), jr.PRNGKey(0))
policy = Controller(Kp=0.4, Ki=0.1, Ks = 0.1)

control_layout = Layout.from_example(jnp.zeros((n_particles, 2)))
control_layout

Layout(treedef=PyTreeDef(*), shapes=((50, 2),), sizes=(100,), offsets=(0,), state_dim=100)

In [32]:
def make_stepper(dt):

    def step_phi(V):
        return to_array(fluid_step(to_field(V), dt))

    def stepper(state, u, t_now, t_next):
        V = state['velocity']
        V = jax.pure_callback(step_phi, jax.ShapeDtypeStruct(V.shape, V.dtype), V, vmap_method="sequential")

        particles = state["particles"]
        new_particles = jax.vmap(advect_particle, in_axes=(None, 0, 0, None))(V, particles, u, dt)
        new = {'velocity': V, 'particles': new_particles}
        return new

    return stepper



step = make_stepper(dt)

initial_condition = DiracInitialCondition(state_0, state_layout=state_layout)
evolution = DiracStateEvolution(F=step, state_layout=state_layout)
def observation_function(state, u, t):   # only the particle positions, not the velocity field
    return state["particles"]

observation_layout = Layout.from_example(observation_function(state_0, None, 0.0))

# observation_model = DiracObservation( observation_function, state_layout=state_layout, 
#                                         observation_layout=observation_layout)
observation_model = GaussianObservation( observation_function, state_layout=state_layout,
                                        observation_layout=observation_layout, cov = 1e-8)


In [33]:
dynamics = DynamicalModel(
        initial_condition=initial_condition,
        state_evolution=evolution,
        observation_model=observation_model,
        state_layout=state_layout,
        observation_layout=observation_layout,
        control_layout=control_layout,
    )

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3788474114.py:4: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [34]:
T = 10.0 
times = jnp.arange(0.0, T, step=dt)

results = dsx.simulate(
    dynamics,
    rng_key=jr.key(13),
    predict_times=times,
    control_policy=policy,
    filter_config=PFConfig(filter_source="cuthbert", record_filtered_states_mean=True, n_particles=1),
    initial_policy_state =s_0
)

/var/folders/r2/j7yw1m1d2p56prg4jm7xprh80000gn/T/ipykernel_7107/3788474114.py:4: RuntimeWarning: jit_compile() not supported by numpy. Running function 'fluid_step' as-is.
  return to_array(fluid_step(to_field(V), dt))


In [35]:
states = results.states
assert all(jnp.isfinite(states[name]).all() for name in states.keys())

In [36]:
# # Video: vorticity + particle trajectory
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
# from IPython.display import HTML
# plt.rcParams["animation.embed_limit"] = 100  # MB
# # LaTeX look without a LaTeX install: matplotlib's Computer Modern fonts.
# # (text.usetex=True needs pdflatex + dvipng, and re-runs LaTeX for every frame.)
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.serif": ["cmr10", "DejaVu Serif"],
#     "mathtext.fontset": "cm",
#     "axes.formatter.use_mathtext": True,
#     "axes.unicode_minus": False,
# })

# V = np.asarray(states["velocity"]).reshape(-1, *state_0["velocity"].shape)   # (T, NX+1, NY+1, 2)
# X = states["particles"][0]# (T, n_particles, 2)
# u_c = 0.5 * (V[:, 1:, :NY, 0] + V[:, :-1, :NY, 0])                           # cell-centred v_x
# v_c = 0.5 * (V[:, :NX, 1:, 1] + V[:, :NX, :-1, 1])                           # cell-centred v_y
# vort = np.gradient(v_c, dx, axis=1) - np.gradient(u_c, dy, axis=2)          # (T, NX, NY)
# vmax = np.percentile(np.abs(vort), 99)

# # Rendering only: upsample the field in space, and optionally in time.
# SMOOTH = 3        # spatial upsampling factor (cubic spline), rendering only
# STEP = 1.0        # stored steps per rendered frame; < 1 interpolates extra frames
# FRAME_DT = float(np.asarray(times)[1] - np.asarray(times)[0]) * STEP   # simulated time per frame
# FPS = 1.0 / FRAME_DT                                                   # real time: 1 s of video per time unit
# from scipy.ndimage import zoom

# def at(i_f, arr):   # linear interpolation in time at a fractional frame index
#     i0 = int(np.floor(i_f)); i1 = min(i0 + 1, len(arr) - 1); a = i_f - i0
#     return (1 - a) * arr[i0] + a * arr[i1]

# fig, ax = plt.subplots(figsize=(10, 5))
# im = ax.imshow(zoom(vort[0], SMOOTH, order=3).T, origin="lower", extent=(0, LX, 0, LY),
#                cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="bilinear")
# ax.add_patch(plt.Circle(cylinder.center.numpy("vector"), float(cylinder.radius), color="k"))
# N_SWARM = X.shape[1]
# SWARM_COLOR = "gold"
# # One line for every trail (NaN-separated) and one for every marker: same colour throughout.
# trail, = ax.plot([], [], "-", lw=0.8, color=SWARM_COLOR, alpha=0.5)
# dots, = ax.plot([], [], "o", color=SWARM_COLOR, mec="k", ms=6, ls="none",
#                 label=rf"swarm ($N = {N_SWARM}$)")
# ax.legend(loc="upper right", fontsize=8)
# fig.colorbar(im, ax=ax, label=r"vorticity $\omega$", shrink=0.8)
# ax.set_xlim(0, LX); ax.set_ylim(0, LY)
# ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$y$")

# def update(i_f):
#     im.set_data(zoom(at(i_f, vort), SMOOTH, order=3).T)
#     x_now, k = at(i_f, X), int(np.ceil(i_f)) + 1
#     seg = np.full((k + 1, N_SWARM, 2), np.nan)      # trailing NaN row separates the trails
#     seg[:k] = X[:k]
#     trail.set_data(seg[..., 0].T.ravel(), seg[..., 1].T.ravel())
#     dots.set_data(x_now[:, 0], x_now[:, 1])
#     ax.set_title(rf"$t = {at(i_f, np.asarray(times)):.1f}$")
#     return (im, trail, dots)

# anim = FuncAnimation(fig, update, frames=np.arange(0, len(X) - 1, STEP),
#                      interval=1000 * FRAME_DT, blit=True)
# plt.close(fig)
# anim.save("wake_swarm.mp4", fps=FPS, dpi=120)   # needs ffmpeg; use a .gif filename for the pillow writer
# HTML(anim.to_jshtml())

In [37]:
state_layout

Layout(treedef=PyTreeDef({'particles': *, 'velocity': *}), shapes=((50, 2), (129, 65, 2)), sizes=(100, 16770), offsets=(0, 100), state_dim=16870)